# Two-dimensional dijet unfolding

Build a flattened $p_{T}^{ave}$–$\eta_{CM}$ response and run a Bayesian RooUnfold closure test using the dedicated eta-dependent JER-default reconstructed distribution, response, misses, and fakes.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.

RooUnfold is loaded separately because only unfolding workflows require it.
Set `ROOUNFOLD_ROOT` when its checkout is not adjacent to this repository.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL
from hist_analysis.python.histogram_io import resolve_combined_file, resolve_direction_file
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, save_canvas, set_2d_style, set_legend_style,
    set_pad_style, set_unfolding_1d_style,
)

# RooUnfold is optional and is initialized only for unfolding notebooks.
ROOUNFOLD_ROOT, ROOUNFOLD_LIBRARY = load_roounfold(
    ROOT,
    project_root=PROJECT_ROOT,
)

from hist_analysis.config.histograms import DIJET_PTAVE_BINS, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.unfolding import (
    UnfoldingInputKeys, as_pt_intervals, build_roounfold_response,
    calculate_response_diagnostics, flatten_pt_eta_projections,
    flatten_sparse_response, load_unfolding_inputs, project_eta_by_pt,
    project_response_eta_blocks, unfold_bayes, write_unfolding_output,
)
from hist_analysis.python.unfolding_plots import (
    draw_flattened_response, draw_projection_response,
    draw_unfolding_classification, draw_unfolding_closure,
    draw_unfolding_closure_by_pt,
)


In [ ]:
# Set the ROOT style
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

In [ ]:
# Define useful functions
text = ROOT.TLatex()
text.SetTextFont(42)
text.SetTextSize(0.04)

# Plot CMS header on the canvas
def plotCMSHeader(collSystem=0, energy=8.16):
    # collSystem: 0 = pp, 1 = pPb, 2 = PbPb
    # energy in TeV
    collSystemStr = "pp" if collSystem == 0 else "pPb" if collSystem == 1 else "PbPb"
    t = ROOT.TLatex()
    t.SetTextFont(42)
    t.SetTextSize(0.05)
    t.DrawLatexNDC(0.15, 0.93, "#bf{CMS} #it{Preliminary}")
    t.SetTextSize(0.04)
    t.DrawLatexNDC(0.6, 0.93, f"{collSystemStr} #sqrt{{s_{{NN}}}} = {energy:.2f} TeV")
    t.SetTextSize(0.05)



## Load RooUnfold

In [ ]:
try:
    import RooUnfold
except ImportError as exc:
    raise ImportError(
        f"Unable to import RooUnfold after loading {ROOUNFOLD_LIBRARY}. "
        f"Check that ROOUNFOLD_ROOT points to the RooUnfold checkout."
    ) from exc

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'Pbgoing'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
# List of eta cuts for analysis
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
# Finite pT-average intervals used for the 2D unfolding. Values outside this range are excluded.
PTAVE_BIN_SET = 'test'  # test or standard
PTAVE_BIN_SETS = {'test': TEST_DIJET_PTAVE_BINS, 'standard': DIJET_PTAVE_BINS}
PT_AVE_BINS = tuple(PTAVE_BIN_SETS[PTAVE_BIN_SET])
ETA_CUT_INDEX = 5
N_ITERATIONS = 4
COMPARISON_TARGET = 'gen'    # gen or ref; response and prior remain Gen-based
PLOT_MISS_AND_FAKES = False
MEASURED_HISTOGRAM_TEMPLATE = 'hRecoDijetPtEtaCMJerDefExtraUnfold_{eta_cut_index}'
MEASURED_LABEL = 'Reco JER def.+#eta-dep.'
RESPONSE_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMVsRecoJerDefExtraPtEtaCM_{eta_cut_index}'
MISS_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMMissJerDefExtra_{eta_cut_index}'
FAKE_HISTOGRAM_TEMPLATE = 'hRecoDijetPtEtaCMFakeJerDefExtra_{eta_cut_index}'
CLASSIFICATION_HISTOGRAM_TEMPLATE = 'hUnfoldingPairClassificationJerDefExtra_{eta_cut_index}'
# Keep weighted response entries above RooUnfold's internal 1e-9 sanitization threshold.
RESPONSE_SCALE = 1.0e12
# RESPONSE_SCALE = 1.0
SAVE_PNG = False

if PTAVE_BIN_SET not in PTAVE_BIN_SETS:
    raise ValueError(f'Unsupported PTAVE_BIN_SET={PTAVE_BIN_SET!r}')
if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError(f'Unsupported GENERATOR={GENERATOR!r}')
if DIRECTION not in ('pgoing', 'Pbgoing', 'combined'):
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
if COMPARISON_TARGET not in ('gen', 'ref'):
    raise ValueError(f'Unsupported COMPARISON_TARGET={COMPARISON_TARGET!r}')
if not isinstance(PLOT_MISS_AND_FAKES, bool):
    raise TypeError('PLOT_MISS_AND_FAKES must be True or False')
if RESPONSE_SCALE <= 0.0:
    raise ValueError(f'RESPONSE_SCALE must be positive, got {RESPONSE_SCALE}')

generator_label = GENERATOR.capitalize()
if DIRECTION == 'combined':
    input_path = resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
else:
    input_path = resolve_direction_file(BASE_DIR, GENERATOR, DIRECTION, FILE_STEM)
eta_cut_tag = f'{ETA_CUTS[ETA_CUT_INDEX]:g}'.replace('.', 'p')
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_UNFOLD2D_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'unfold2D',
))
OUTPUT_TAG = f'{GENERATOR}_{DIRECTION}_unfold2D_jerDefExtra_to_{COMPARISON_TARGET}_eta_{eta_cut_tag}_iter_{N_ITERATIONS}'
OUTPUT_ROOT_FILE = OUTPUT_DIR / f'{OUTPUT_TAG}.root'
eta_cuts = ETA_CUTS
pt_ave_bins = as_pt_intervals(PT_AVE_BINS)

generator_label

In [ ]:
from hist_analysis.python.histogram_io import load_histogram
n_eta_cuts = len(eta_cuts)
eta_idx = ETA_CUT_INDEX
keys = UnfoldingInputKeys(
    truth=f'hGenDijetPtEtaCM_{eta_idx}',
    measured=MEASURED_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    response=RESPONSE_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    miss=MISS_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    fake=FAKE_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    classification=CLASSIFICATION_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
)
inputs = load_unfolding_inputs(input_path, keys)
genPtEtaCM=[None]*n_eta_cuts; genPtEtaCM[eta_idx]=inputs.truth
refPtEtaCM=[None]*n_eta_cuts; refPtEtaCM[eta_idx]=load_histogram(str(input_path),f'hRefDijetPtEtaCM_{eta_idx}')
recoPtEtaCM=[None]*n_eta_cuts; recoPtEtaCM[eta_idx]=inputs.measured
genPtEtaCMMiss=[None]*n_eta_cuts; genPtEtaCMMiss[eta_idx]=inputs.miss
recoPtEtaCMFake=[None]*n_eta_cuts; recoPtEtaCMFake[eta_idx]=inputs.fake
genPtEtaCMVsRecoPtEtaCM=[None]*n_eta_cuts; genPtEtaCMVsRecoPtEtaCM[eta_idx]=inputs.response
unfoldingPairClassification=[None]*n_eta_cuts; unfoldingPairClassification[eta_idx]=inputs.classification
print(input_path); print(keys)

In [ ]:
classification_canvas, classification = draw_unfolding_classification(
    unfoldingPairClassification[ETA_CUT_INDEX],
    annotations=(generator_label, f'|#eta_{{CM}}^{{jet}}| < {ETA_CUTS[ETA_CUT_INDEX]:g}', MEASURED_LABEL),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_pair_classification.pdf',
    save_png=SAVE_PNG, grid=False,
    canvas_name='canvas_unfolding_classification_jerDefExtra',
)
classification_canvas

In [ ]:
# # For test purpose only, plot the first histogram

# if not genPtEtaCM:
#     raise RuntimeError("genPtEtaCM is empty; load the ROOT file first")

# canvas_name = "canvas_gen_pt_eta_cm_0"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_gen_dijet_pt_eta_cm_0 = ROOT.TCanvas(canvas_name, "cGenDijetPtEtaCM_0", 800, 800)
# canvas_gen_dijet_pt_eta_cm_0.cd()
# set_pad_style(ROOT.gPad, grid_x=False, grid_y=False)
# ROOT.gPad.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
# set_2d_style(genPtEtaCM[0])
# genPtEtaCM[0].Draw("COLZ")
# canvas_gen_dijet_pt_eta_cm_0.Modified()
# canvas_gen_dijet_pt_eta_cm_0.Update()
# canvas_gen_dijet_pt_eta_cm_0

In [ ]:
genEtaCM=[None]*n_eta_cuts; genEtaCM[eta_idx]=project_eta_by_pt(genPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hGenDijetEtaCM_{eta_idx}')
refEtaCM=[None]*n_eta_cuts; refEtaCM[eta_idx]=project_eta_by_pt(refPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hRefDijetEtaCM_{eta_idx}')
recoEtaCM=[None]*n_eta_cuts; recoEtaCM[eta_idx]=project_eta_by_pt(recoPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hRecoDijetEtaCM_{eta_idx}')
genEtaCMMiss=[None]*n_eta_cuts; genEtaCMMiss[eta_idx]=project_eta_by_pt(genPtEtaCMMiss[eta_idx],pt_ave_bins,name_prefix=f'hGenDijetEtaCMMiss_{eta_idx}')
recoEtaCMFake=[None]*n_eta_cuts; recoEtaCMFake[eta_idx]=project_eta_by_pt(recoPtEtaCMFake[eta_idx],pt_ave_bins,name_prefix=f'hRecoDijetEtaCMFake_{eta_idx}')
gen2recoResponse=[None]*n_eta_cuts; gen2recoResponse[eta_idx]=project_response_eta_blocks(genPtEtaCMVsRecoPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hGen2RecoDijetEtaCM_{eta_idx}')

In [ ]:
# # Plot all genEtaCM projections (all pt_ave bins) on one canvas for a given eta cut
# if not genEtaCM:
#     raise RuntimeError("genEtaCM is empty; run the projection cell first")

# eta_idx = 5  # choose eta-cut index here
# if eta_idx < 0 or eta_idx >= len(eta_cuts):
#     raise IndexError(f"eta_idx={eta_idx} is out of range for eta_cuts")

# canvas_name = f"canvas_genEtaCM_allPt_eta{eta_idx}"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_genEtaCM_allPt = ROOT.TCanvas(canvas_name, f"Gen etaCM projections (eta idx {eta_idx})", 800, 800)
# canvas_genEtaCM_allPt.cd()
# set_pad_style(ROOT.gPad, grid_x=False, grid_y=False)

# gen_eta_overlays = []
# max_val = 0.0
# for pt_idx in range(len(pt_ave_bins) - 1):
#     hist = genEtaCM[eta_idx][pt_idx].Clone(f"hGenEtaCM_overlay_eta{eta_idx}_pt{pt_idx}")
#     hist.SetDirectory(0)
#     set_unfolding_1d_style(hist, 'gen')
#     integral = hist.Integral()
#     if integral > 0:
#         hist.Scale(1.0 / integral)
#     hist.SetTitle(";#eta_{CM};Normalized entries")
#     draw_opt = "E1" if pt_idx == 0 else "E1 SAME"
#     hist.Draw(draw_opt)
#     gen_eta_overlays.append(hist)
#     max_val = max(max_val, hist.GetMaximum())

# legend = ROOT.TLegend(0.6, 0.75, 0.88, 0.88)
# set_legend_style(legend)
# legend.SetTextFont(42)
# legend.SetTextSize(0.03)
# for pt_idx in range(len(pt_ave_bins) - 1):
#     label = f"{pt_ave_bins[pt_idx]} < p_{{T}}^{{ave}} < {pt_ave_bins[pt_idx + 1]} GeV"
#     legend.AddEntry(gen_eta_overlays[pt_idx], label, "p")
# legend.Draw()

# text = ROOT.TLatex()
# text.SetNDC(True)
# text.SetTextFont(42)
# text.SetTextSize(0.04)
# text.DrawLatex(0.16, 0.92, f"Gen #eta_{{CM}} projections, eta-cut index = {eta_idx}")

# canvas_genEtaCM_allPt.Modified()
# canvas_genEtaCM_allPt.Update()
# canvas_genEtaCM_allPt

In [ ]:
eta_idx = ETA_CUT_INDEX
projection_response_canvases = []
for pt_bin_idx, (pt_low, pt_high) in enumerate(pt_ave_bins):
    spectra = {
        'gen': ('Gen', genEtaCM[eta_idx][pt_bin_idx]),
        'reco': (MEASURED_LABEL, recoEtaCM[eta_idx][pt_bin_idx]),
        'miss': ('Miss', genEtaCMMiss[eta_idx][pt_bin_idx]),
        'fake': ('Fake', recoEtaCMFake[eta_idx][pt_bin_idx]),
    }
    if COMPARISON_TARGET == 'ref':
        spectra = {'gen': spectra['gen'], 'ref': ('Ref comparison target', refEtaCM[eta_idx][pt_bin_idx]), **{k: v for k, v in spectra.items() if k != 'gen'}}
    canvas = draw_projection_response(
        spectra, gen2recoResponse[eta_idx][pt_bin_idx],
        eta_range=(-eta_cuts[eta_idx]-0.1, eta_cuts[eta_idx]+0.1),
        response_titles=(f'{MEASURED_LABEL} #eta_{{CM}}', 'Gen #eta_{CM}'),
        annotations=(generator_label, f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV', f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}', DIJET_DELTA_PHI_SELECTION_LABEL),
        plot_miss_and_fakes=PLOT_MISS_AND_FAKES,
        output=OUTPUT_DIR / f'{OUTPUT_TAG}_projection_response_pt_{pt_low:g}_{pt_high:g}.pdf',
        save_png=SAVE_PNG, canvas_name=f'canvas_projection_response_ptBin{pt_bin_idx}',
    )
    projection_response_canvases.append(canvas)
projection_response_canvases

In [ ]:
eta_idx = ETA_CUT_INDEX
hGenTruthEtaCM, layout = flatten_pt_eta_projections(genEtaCM[eta_idx], name='hGenTruthEtaCM', pt_bins=pt_ave_bins)
hRefComparisonEtaCM, _ = flatten_pt_eta_projections(refEtaCM[eta_idx], name='hRefComparisonEtaCM', layout=layout)
hRecoMeasuredEtaCM, _ = flatten_pt_eta_projections(recoEtaCM[eta_idx], name='hRecoMeasuredEtaCM', layout=layout)
hGenTruthEtaCMMiss, _ = flatten_pt_eta_projections(genEtaCMMiss[eta_idx], name='hGenTruthEtaCMMiss', layout=layout)
hRecoMeasuredEtaCMFake, _ = flatten_pt_eta_projections(recoEtaCMFake[eta_idx], name='hRecoMeasuredEtaCMFake', layout=layout)
hResponseEtaCM, _ = flatten_sparse_response(genPtEtaCMVsRecoPtEtaCM[eta_idx], pt_ave_bins, name='hResponseEtaCM', layout=layout)
nPtSelections, nEtaBins, nGlobalBins = layout.n_pt_bins, layout.n_eta_bins, layout.n_global_bins
comparisonTargetLabel = 'Gen' if COMPARISON_TARGET == 'gen' else 'Ref'
hComparisonTargetEtaCM = hGenTruthEtaCM if COMPARISON_TARGET == 'gen' else hRefComparisonEtaCM
print(f'nPtSelections={nPtSelections}, nEtaBins={nEtaBins}, nGlobalBins={nGlobalBins}')

In [ ]:
flattened_spectra = {
    'gen': ('Gen', hGenTruthEtaCM),
    'reco': (MEASURED_LABEL, hRecoMeasuredEtaCM),
    'miss': ('Miss', hGenTruthEtaCMMiss),
    'fake': ('Fake', hRecoMeasuredEtaCMFake),
}
if COMPARISON_TARGET == 'ref':
    flattened_spectra = {'gen': flattened_spectra['gen'], 'ref': ('Ref comparison target', hRefComparisonEtaCM), **{k: v for k, v in flattened_spectra.items() if k != 'gen'}}
canvas_flattened = draw_flattened_response(
    flattened_spectra, hResponseEtaCM, annotations=(generator_label, f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}'),
    plot_miss_and_fakes=PLOT_MISS_AND_FAKES, output=OUTPUT_DIR / f'{OUTPUT_TAG}_flattened_response.pdf',
    save_png=SAVE_PNG, canvas_name='canvas_flattened',
)
canvas_flattened

In [ ]:
diagnostics = calculate_response_diagnostics(
    hResponseEtaCM, hGenTruthEtaCM, hRecoMeasuredEtaCM,
    explicit_miss=hGenTruthEtaCMMiss, explicit_fake=hRecoMeasuredEtaCMFake,
)
hMatchedTruthEtaCM = diagnostics.matched_truth
hMatchedRecoEtaCM = diagnostics.matched_measured
hEffectiveMissEtaCM = diagnostics.effective_miss
hEffectiveFakeEtaCM = diagnostics.effective_fake
hBoundaryMissEtaCM = diagnostics.boundary_miss
hBoundaryFakeEtaCM = diagnostics.boundary_fake
response_bundle = build_roounfold_response(
    RooUnfold, hGenTruthEtaCM, hRecoMeasuredEtaCM, hResponseEtaCM,
    diagnostics=diagnostics, scale=RESPONSE_SCALE, require_fakes=True,
)
response = response_bundle.response
unfolding_result = unfold_bayes(
    RooUnfold, response_bundle, hRecoMeasuredEtaCM, iterations=N_ITERATIONS,
    handle_fakes=True, name='hUnfoldedEtaCM',
)
unfold = unfolding_result.algorithm
hUnfoldedEtaCM = unfolding_result.histogram
covariance_matrix = unfolding_result.covariance
print(f'Explicit misses: {hGenTruthEtaCMMiss.Integral():.6g}; effective: {hEffectiveMissEtaCM.Integral():.6g}')
print(f'Explicit fakes: {hRecoMeasuredEtaCMFake.Integral():.6g}; effective: {hEffectiveFakeEtaCM.Integral():.6g}')

In [ ]:
canvas_unfolded, hMeasuredToTruth, hUnfoldedToTruth = draw_unfolding_closure(
    hComparisonTargetEtaCM, hRecoMeasuredEtaCM, hUnfoldedEtaCM,
    target_label=comparisonTargetLabel, target_role=COMPARISON_TARGET, measured_label=MEASURED_LABEL,
    x_title='global #eta_{CM} bin', ratio_range=(0.5, 2.5),
    annotations=(generator_label, f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}'),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_closure.pdf', save_png=SAVE_PNG,
    canvas_name='canvas_unfolded', ratio_name_prefix='hFlattenedClosure',
)
canvas_unfolded

## Unfolded eta distributions in all pTave intervals

Extract every pTave block from the flattened unfolded histogram. For each interval, compare unfolded and eta-dependent JER-default reco eta distributions with the configured Gen or Ref comparison target. The response and Bayesian prior remain Gen-based for both choices.

In [ ]:
target_by_pt = genEtaCM[eta_idx] if COMPARISON_TARGET == 'gen' else refEtaCM[eta_idx]
(hUnfoldedEtaCMByPt, hRecoToComparisonEtaCMByPt,
 hUnfoldedToComparisonEtaCMByPt, unfolded_eta_canvases) = draw_unfolding_closure_by_pt(
    hUnfoldedEtaCM, target_by_pt, recoEtaCM[eta_idx], layout,
    output_dir=OUTPUT_DIR, output_tag=OUTPUT_TAG, target_label=comparisonTargetLabel,
    target_role=COMPARISON_TARGET, measured_label=MEASURED_LABEL,
    eta_range=(-eta_cuts[eta_idx]-0.1, eta_cuts[eta_idx]+0.1), ratio_range=(0.5, 1.5),
    annotation_prefix=(generator_label, f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}'), save_png=SAVE_PNG,
)
hGenEtaCMByPt = [hist.Clone(f'hGenEtaCM_ptBin{i}') for i, hist in enumerate(genEtaCM[eta_idx])]
hRefEtaCMByPt = [hist.Clone(f'hRefEtaCM_ptBin{i}') for i, hist in enumerate(refEtaCM[eta_idx])]
hRecoEtaCMByPt = [hist.Clone(f'hRecoEtaCM_ptBin{i}') for i, hist in enumerate(recoEtaCM[eta_idx])]
unfolded_eta_canvases

## Save unfolding output

Write the flattened spectra, response diagnostics, unfolded result, ratios, covariance matrix, and configuration metadata to `hist_analysis/output/unfold2D/`.

In [ ]:
output_histograms = (
    hGenTruthEtaCM, hRefComparisonEtaCM, hRecoMeasuredEtaCM, hGenTruthEtaCMMiss, hRecoMeasuredEtaCMFake, hResponseEtaCM,
    hMatchedTruthEtaCM, hMatchedRecoEtaCM, hEffectiveMissEtaCM, hEffectiveFakeEtaCM, hBoundaryMissEtaCM, hBoundaryFakeEtaCM,
    hUnfoldedEtaCM, hMeasuredToTruth, hUnfoldedToTruth, *hGenEtaCMByPt, *hRefEtaCMByPt, *hRecoEtaCMByPt,
    *hUnfoldedEtaCMByPt, *hRecoToComparisonEtaCMByPt, *hUnfoldedToComparisonEtaCMByPt,
)
write_unfolding_output(
    OUTPUT_ROOT_FILE, histograms=output_histograms, covariance=covariance_matrix, response=response,
    metadata={'generator': GENERATOR, 'direction': DIRECTION, 'eta_cut': eta_cuts[eta_idx], 'pt_ave_bins': pt_ave_bins,
              'ptave_bin_set': PTAVE_BIN_SET, 'iterations': N_ITERATIONS, 'comparison_target': COMPARISON_TARGET,
              'plot_miss_and_fakes': PLOT_MISS_AND_FAKES, 'response_scale': RESPONSE_SCALE, 'handle_fakes': True},
)
print(f'Wrote unfolding output to {OUTPUT_ROOT_FILE}')